In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers sentencepiece accelerate

In [ ]:

import pandas as pd
import numpy as np

print("Libraries ready.")

Libraries ready.


In [ ]:
# Load the fixed company-level splits
train = pd.read_csv(
    "/content/drive/MyDrive/Fraud_Project/E1_train.csv"
)

validation = pd.read_csv(
    "/content/drive/MyDrive/Fraud_Project/E1_validation.csv"
)

test = pd.read_csv(
    "/content/drive/MyDrive/Fraud_Project/E1_test.csv"
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print("\nRequired columns:")
print("CIK:", "cik" in train.columns)
print("Fiscal year:", "fiscal_year" in train.columns)
print("MD&A:", "mda" in train.columns)
print("Fraud label:", "is_fraud" in train.columns)

Train: (18734, 23)
Validation: (6322, 23)
Test: (6185, 23)

Required columns:
CIK: True
Fiscal year: True
MD&A: True
Fraud label: True


In [ ]:
# Check MD&A quality in each split

for name, data in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{'='*50}")
    print(f"{name} MD&A CHECK")
    print(f"{'='*50}")

    print("Missing MD&A:", data["mda"].isna().sum())
    print("Empty MD&A:", (data["mda"].fillna("").str.strip() == "").sum())

    print("\nMD&A length statistics:")
    print(data["mda"].fillna("").str.len().describe())

    print("\nDuplicate CIK-year pairs:",
          data.duplicated(subset=["cik", "fiscal_year"]).sum())

    print("\nFraud labels:")
    print(data["is_fraud"].value_counts().sort_index())


Train MD&A CHECK
Missing MD&A: 0
Empty MD&A: 0

MD&A length statistics:
count    1.873400e+04
mean     8.194024e+04
std      5.975668e+04
min      3.000000e+02
25%      4.620525e+04
50%      7.050050e+04
75%      1.030540e+05
max      1.251239e+06
Name: mda, dtype: float64

Duplicate CIK-year pairs: 0

Fraud labels:
is_fraud
0    18579
1      155
Name: count, dtype: int64

Validation MD&A CHECK
Missing MD&A: 0
Empty MD&A: 0

MD&A length statistics:
count    6.322000e+03
mean     8.025561e+04
std      5.865961e+04
min      3.090000e+02
25%      4.755275e+04
50%      7.052700e+04
75%      9.840825e+04
max      1.018864e+06
Name: mda, dtype: float64

Duplicate CIK-year pairs: 0

Fraud labels:
is_fraud
0    6265
1      57
Name: count, dtype: int64

Test MD&A CHECK
Missing MD&A: 0
Empty MD&A: 0

MD&A length statistics:
count      6185.000000
mean      77019.243007
std       53786.604573
min         301.000000
25%       42192.000000
50%       66157.000000
75%       99186.000000
max      446

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "facebook/bart-large-mnli"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Maximum model length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded successfully.
Maximum model length: 1024


In [ ]:
MAX_CHUNK_TOKENS = 900
CHUNK_OVERLAP = 100

def chunk_mda(text, tokenizer,
              max_tokens=MAX_CHUNK_TOKENS,
              overlap=CHUNK_OVERLAP):

    # Tokenize without triggering the model's max-length warning
    token_ids = tokenizer(
        str(text),
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    chunks = []

    step = max_tokens - overlap

    for start in range(0, len(token_ids), step):

        chunk_ids = token_ids[start:start + max_tokens]

        if not chunk_ids:
            continue

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        chunks.append(chunk_text)

        if start + max_tokens >= len(token_ids):
            break

    return chunks


# Test on one MD&A
sample_chunks = chunk_mda(
    train.loc[0, "mda"],
    tokenizer
)

print("Original token count:",
      len(tokenizer(
          train.loc[0, "mda"],
          add_special_tokens=False,
          truncation=False
      )["input_ids"]))

print("Number of chunks:", len(sample_chunks))

print("First chunk characters:",
      len(sample_chunks[0]))

print("\nFirst chunk preview:")
print(sample_chunks[0][:500])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (19805 > 1024). Running this sequence through the model will result in indexing errors


Original token count: 19805
Number of chunks: 25
First chunk characters: 4593

First chunk preview:
 Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations 

The following discussion should be read in conjunction with the consolidated financial statements and notes thereto included elsewhere in this Annual Report on Form 10-K. This report contains forward-looking statements including, without limitation, statements regarding trends, seasonality, cyclicality and growth in, and drivers of, the markets we sell into, our strategic direction, our future effect


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Make sure text is string
train_text = train["mda"].fillna("").astype(str)
val_text = validation["mda"].fillna("").astype(str)
test_text = test["mda"].fillna("").astype(str)

# TF-IDF fitted ONLY on training text
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    max_features=50000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(train_text)
X_val_tfidf = tfidf.transform(val_text)
X_test_tfidf = tfidf.transform(test_text)

print("TF-IDF created successfully.")
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

TF-IDF created successfully.
Train: (18734, 50000)
Validation: (6322, 50000)
Test: (6185, 50000)
Vocabulary size: 50000


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# E2 baseline: TF-IDF + Logistic Regression
lr_tfidf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="liblinear",
    random_state=42
)

lr_tfidf.fit(X_train_tfidf, train["is_fraud"])

# Validation probabilities
val_prob = lr_tfidf.predict_proba(X_val_tfidf)[:, 1]

print("E2 TF-IDF + Logistic Regression trained successfully.")
print("Validation probability range:")
print("Min:", val_prob.min())
print("Max:", val_prob.max())
print("Mean:", val_prob.mean())
print("Median:", np.median(val_prob))

print("\nValidation ROC-AUC:",
      roc_auc_score(validation["is_fraud"], val_prob))

print("Validation PR-AUC:",
      average_precision_score(validation["is_fraud"], val_prob))

E2 TF-IDF + Logistic Regression trained successfully.
Validation probability range:
Min: 0.0028718279080206252
Max: 0.5275685558832942
Mean: 0.05345414172672768
Median: 0.03793228893835664

Validation ROC-AUC: 0.6661598129401717
Validation PR-AUC: 0.017303537508392614


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [
    0.50,
    0.30,
    0.20,
    0.15,
    0.10,
    0.075,
    0.05,
    0.04,
    0.03,
    0.02,
    0.01
]

threshold_results = []

y_val = validation["is_fraud"].values

for threshold in thresholds:
    y_pred = (val_prob >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    threshold_results.append({
        "threshold": threshold,
        "predicted_fraud": int(y_pred.sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

best_row = threshold_df.loc[threshold_df["f1"].idxmax()]

print("\nBest validation threshold by F1:")
print(best_row)

 threshold  predicted_fraud  precision   recall       f1
     0.500                1   0.000000 0.000000 0.000000
     0.300               36   0.000000 0.000000 0.000000
     0.200              142   0.014085 0.035088 0.020101
     0.150              302   0.013245 0.070175 0.022284
     0.100              791   0.022756 0.315789 0.042453
     0.075             1373   0.019665 0.473684 0.037762
     0.050             2383   0.014268 0.596491 0.027869
     0.040             3009   0.012961 0.684211 0.025440
     0.030             3785   0.012153 0.807018 0.023946
     0.020             4820   0.010788 0.912281 0.021325
     0.010             5941   0.009594 1.000000 0.019006

Best validation threshold by F1:
threshold            0.100000
predicted_fraud    791.000000
precision            0.022756
recall               0.315789
f1                   0.042453
Name: 4, dtype: float64


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Final test probabilities
test_prob = lr_tfidf.predict_proba(X_test_tfidf)[:, 1]

# Threshold selected ONLY from validation
E2_THRESHOLD = 0.10

test_pred = (test_prob >= E2_THRESHOLD).astype(int)

# Metrics
test_precision = precision_score(
    test["is_fraud"], test_pred, zero_division=0
)

test_recall = recall_score(
    test["is_fraud"], test_pred, zero_division=0
)

test_f1 = f1_score(
    test["is_fraud"], test_pred, zero_division=0
)

test_roc_auc = roc_auc_score(
    test["is_fraud"], test_prob
)

test_pr_auc = average_precision_score(
    test["is_fraud"], test_prob
)

cm = confusion_matrix(test["is_fraud"], test_pred)

print("===== E2 TF-IDF + Logistic Regression: FINAL TEST =====")
print(f"Threshold: {E2_THRESHOLD}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")
print(f"PR-AUC:    {test_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nPredicted fraud cases:", int(test_pred.sum()))
print("Actual fraud cases:", int(test["is_fraud"].sum()))

===== E2 TF-IDF + Logistic Regression: FINAL TEST =====
Threshold: 0.1
Precision: 0.0309
Recall:    0.3438
F1-score:  0.0566
ROC-AUC:   0.7300
PR-AUC:    0.0321

Confusion Matrix:
[[5430  691]
 [  42   22]]

Predicted fraud cases: 713
Actual fraud cases: 64


In [ ]:
import joblib
import os

SAVE_DIR = "/content/drive/MyDrive/Fraud_Project"

# Save TF-IDF vectorizer
joblib.dump(
    tfidf,
    os.path.join(SAVE_DIR, "E2_TFIDF_vectorizer.pkl")
)

# Save Logistic Regression model
joblib.dump(
    lr_tfidf,
    os.path.join(SAVE_DIR, "E2_TFIDF_LogisticRegression_model.pkl")
)

# Save selected threshold
with open(
    os.path.join(SAVE_DIR, "E2_TFIDF_threshold.txt"),
    "w"
) as f:
    f.write(str(E2_THRESHOLD))

print("E2 artifacts saved successfully.")
print(os.path.join(SAVE_DIR, "E2_TFIDF_vectorizer.pkl"))
print(os.path.join(SAVE_DIR, "E2_TFIDF_LogisticRegression_model.pkl"))
print(os.path.join(SAVE_DIR, "E2_TFIDF_threshold.txt"))

E2 artifacts saved successfully.
/content/drive/MyDrive/Fraud_Project/E2_TFIDF_vectorizer.pkl
/content/drive/MyDrive/Fraud_Project/E2_TFIDF_LogisticRegression_model.pkl
/content/drive/MyDrive/Fraud_Project/E2_TFIDF_threshold.txt


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# E2: TF-IDF + MLP
mlp_e2 = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    random_state=42
)

mlp_e2.fit(
    X_train_tfidf,
    train["is_fraud"]
)

# Validation probabilities
e2_mlp_val_prob = mlp_e2.predict_proba(X_val_tfidf)[:, 1]

print("E2 TF-IDF + MLP trained successfully.")
print("Iterations:", mlp_e2.n_iter_)

print("\nValidation probability range:")
print("Min:", e2_mlp_val_prob.min())
print("Max:", e2_mlp_val_prob.max())
print("Mean:", e2_mlp_val_prob.mean())
print("Median:", np.median(e2_mlp_val_prob))

print("\nValidation ROC-AUC:",
      roc_auc_score(validation["is_fraud"], e2_mlp_val_prob))

print("Validation PR-AUC:",
      average_precision_score(validation["is_fraud"], e2_mlp_val_prob))

E2 TF-IDF + MLP trained successfully.
Iterations: 34

Validation probability range:
Min: 5.906723352273633e-07
Max: 0.17843815990774603
Mean: 0.00026783996712323715
Median: 3.456038348803948e-05

Validation ROC-AUC: 0.6639979837862814
Validation PR-AUC: 0.02480727018649871


In [ ]:
thresholds = [
    0.10,
    0.075,
    0.05,
    0.03,
    0.02,
    0.01,
    0.005,
    0.002,
    0.001,
    0.0005,
    0.0001,
    0.00005,
    0.00001
]

e2_mlp_threshold_results = []

y_val = validation["is_fraud"].values

for threshold in thresholds:
    y_pred = (e2_mlp_val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val, y_pred, zero_division=0
    )
    recall = recall_score(
        y_val, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_val, y_pred, zero_division=0
    )

    e2_mlp_threshold_results.append({
        "threshold": threshold,
        "predicted_fraud": int(y_pred.sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

e2_mlp_threshold_df = pd.DataFrame(
    e2_mlp_threshold_results
)

print(e2_mlp_threshold_df.to_string(index=False))

best_e2_mlp_row = e2_mlp_threshold_df.loc[
    e2_mlp_threshold_df["f1"].idxmax()
]

print("\nBest E2 MLP validation threshold by F1:")
print(best_e2_mlp_row)


 threshold  predicted_fraud  precision   recall       f1
   0.10000                1   0.000000 0.000000 0.000000
   0.07500                1   0.000000 0.000000 0.000000
   0.05000                1   0.000000 0.000000 0.000000
   0.03000                2   0.000000 0.000000 0.000000
   0.02000                9   0.000000 0.000000 0.000000
   0.01000               28   0.000000 0.000000 0.000000
   0.00500               70   0.042857 0.052632 0.047244
   0.00200              109   0.082569 0.157895 0.108434
   0.00100              187   0.053476 0.175439 0.081967
   0.00050              325   0.040000 0.228070 0.068063
   0.00010             1416   0.014831 0.368421 0.028513
   0.00005             2475   0.014545 0.631579 0.028436
   0.00001             5238   0.010500 0.964912 0.020774

Best E2 MLP validation threshold by F1:
threshold            0.002000
predicted_fraud    109.000000
precision            0.082569
recall               0.157895
f1                   0.108434
Name: 7, dt

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Test probabilities
e2_mlp_test_prob = mlp_e2.predict_proba(X_test_tfidf)[:, 1]

# Threshold selected ONLY from validation
E2_MLP_THRESHOLD = 0.002

e2_mlp_test_pred = (
    e2_mlp_test_prob >= E2_MLP_THRESHOLD
).astype(int)

# Test metrics
e2_mlp_test_precision = precision_score(
    test["is_fraud"],
    e2_mlp_test_pred,
    zero_division=0
)

e2_mlp_test_recall = recall_score(
    test["is_fraud"],
    e2_mlp_test_pred,
    zero_division=0
)

e2_mlp_test_f1 = f1_score(
    test["is_fraud"],
    e2_mlp_test_pred,
    zero_division=0
)

e2_mlp_test_roc_auc = roc_auc_score(
    test["is_fraud"],
    e2_mlp_test_prob
)

e2_mlp_test_pr_auc = average_precision_score(
    test["is_fraud"],
    e2_mlp_test_prob
)

e2_mlp_cm = confusion_matrix(
    test["is_fraud"],
    e2_mlp_test_pred
)

print("===== E2 TF-IDF + MLP: FINAL TEST =====")
print(f"Threshold: {E2_MLP_THRESHOLD}")
print(f"Precision: {e2_mlp_test_precision:.4f}")
print(f"Recall:    {e2_mlp_test_recall:.4f}")
print(f"F1-score:  {e2_mlp_test_f1:.4f}")
print(f"ROC-AUC:   {e2_mlp_test_roc_auc:.4f}")
print(f"PR-AUC:    {e2_mlp_test_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(e2_mlp_cm)

print("\nPredicted fraud cases:",
      int(e2_mlp_test_pred.sum()))

print("Actual fraud cases:",
      int(test["is_fraud"].sum()))


===== E2 TF-IDF + MLP: FINAL TEST =====
Threshold: 0.002
Precision: 0.0317
Recall:    0.0625
F1-score:  0.0421
ROC-AUC:   0.6491
PR-AUC:    0.0198

Confusion Matrix:
[[5999  122]
 [  60    4]]

Predicted fraud cases: 126
Actual fraud cases: 64


In [ ]:
import joblib
import os

SAVE_DIR = "/content/drive/MyDrive/Fraud_Project"

# Save MLP model
joblib.dump(
    mlp_e2,
    os.path.join(SAVE_DIR, "E2_TFIDF_MLP_model.pkl")
)

# Save validation-selected threshold
with open(
    os.path.join(SAVE_DIR, "E2_TFIDF_MLP_threshold.txt"),
    "w"
) as f:
    f.write(str(E2_MLP_THRESHOLD))

print("E2 MLP artifacts saved successfully.")
print(os.path.join(SAVE_DIR, "E2_TFIDF_MLP_model.pkl"))
print(os.path.join(SAVE_DIR, "E2_TFIDF_MLP_threshold.txt"))

E2 MLP artifacts saved successfully.
/content/drive/MyDrive/Fraud_Project/E2_TFIDF_MLP_model.pkl
/content/drive/MyDrive/Fraud_Project/E2_TFIDF_MLP_threshold.txt
